<a href="https://colab.research.google.com/github/inpersonin/nlp_text_classification_tfidf_xlmr.ipynb/blob/main/nlp_text_classification_tfidf_xlmr.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import os

In [2]:
import kagglehub
path = kagglehub.dataset_download("studiousmonk/code-mixed-hinglish-abusive-and-hate-speech")

print("Path to dataset files:", path)

100%|██████████| 14.3k/14.3k [00:00<00:00, 25.2MB/s]

Extracting files...
Path to dataset files: /root/.cache/kagglehub/datasets/studiousmonk/code-mixed-hinglish-abusive-and-hate-speech/versions/1


In [3]:
data_file_name = "train.csv"
full_data_path = os.path.join(path, data_file_name)

if os.path.exists(full_data_path):
    df = pd.read_csv(full_data_path)
    print(f"Successfully loaded '{data_file_name}' into a DataFrame.")
    display(df.head())
    display(df.tail())
    display(df.columns)
    print(df['label'].value_counts())
else:
    print(f"Error: The expected data file '{data_file_name}' was not found in the downloaded directory.")
    print(f"Contents of the downloaded directory '{path}':")
    for item in os.listdir(path):
        print(f"- {item}")
    df = None

Successfully loaded 'train.csv' into a DataFrame.


,text,label
0,Thank you so much bhai,not offensive
1,bhen ke lode,offensive
2,bh3n k3 l0d3,offensive
3,madarchod saale,offensive
4,madrch0d harami,offensive


,text,label
1361,bhai tu toh legend hai legend,not offensive
1362,tu toh dil se dil tak jaata hai yaar,not offensive
1363,arey tu toh mera real bhai hai,not offensive
1364,tu toh hamesha yaad rahega bhai,not offensive
1365,bhai tu toh best hai mere liye,not offensive


Index(['text', 'label'], dtype='object')

label
offensive        695
not offensive    671
Name: count, dtype: int64


In [4]:
pd.set_option('display.max_colwidth', None)

print(df[df['label'] == 'offensive'].sample(30, random_state=42)['text'].tolist())

print(df[df['label'] == 'not offensive'].sample(30, random_state=42)['text'].tolist())

['saale tharki insaan', 'tere baap ko napunsak bana ke Fair & Lovely ka ad shoot karwa dunga', 'kutte ki dum tedhi', 'l*nd st@nd!ng', 'kutte ki jhaant', 'bhadwa dalal', 'lund le madarchod', 'l0d3 k3 b@@l', 'behen ka yaar ban gaya', 'g@@nd ph0d d00ng@', 't3ri b3hn k@ bh0sd@', 'jhantu ka jhanda', 'tere baap ka lund kaat ke pencil bana ke tere exam mein use karunga', 'l*nd chaba ja', 'chodu company hai', 'gandu plans', 'saale tharki dreams', 'teri behen ke saath Oyo mein 24 ghante non-stop pelunga', 'teri maa ki gaand mein rocket daal ke SpaceX ko bech dunga', 'teri behen ko nanga karke Jantar Mantar pe protest karwa dunga', 'teri maa ko randi bana ke uska rate card print karwa dunga', 'bh3n k@ l0v3r', 'g@ndu b@nn@ b@nd', 'g@ndu str@t3gy', 'tere baap ke gaand mein Taj Mahal ghusa dunga', 'randwe ka business', 't3ri m@ k! ch*t', 'gaand mein danda', 'teri maa ko randi bana ke Lal Quile pe chodunga', 'ch*t ph@tw@ r@h@']
['tu toh full bakbak hi karta hai', 'bhai tu toh full shy type hai', 'tu

In [5]:
print(df.duplicated(subset='text').sum())
print(df['text'].str.len().describe())

30
count    1366.000000
mean       33.385066
std        17.535761
min         8.000000
25%        17.000000
50%        31.000000
75%        49.000000
max        81.000000
Name: text, dtype: float64


In [6]:
df = df.drop_duplicates(subset='text').reset_index(drop=True)

In [7]:
import re

def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z0-9@!*\s]', '', text)
    text = re.sub(r'\s+', ' ', text).strip()
    return text

df['clean_text'] = df['text'].apply(clean_text)

In [8]:
from sklearn.model_selection import train_test_split

df['label_num'] = df['label'].map({'not offensive': 0, 'offensive': 1})

X_train, X_test, y_train, y_test = train_test_split(
    df['clean_text'], df['label_num'],
    test_size=0.2, random_state=42, stratify=df['label_num']
)

In [9]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report

tfidf_word = TfidfVectorizer(ngram_range=(1,2), min_df=2)
Xtr_word = tfidf_word.fit_transform(X_train)
Xte_word = tfidf_word.transform(X_test)

clf_word = LogisticRegression(max_iter=1000)
clf_word.fit(Xtr_word, y_train)
print("WORD-LEVEL TF-IDF")
print(classification_report(y_test, clf_word.predict(Xte_word)))

tfidf_char = TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5), min_df=2)
Xtr_char = tfidf_char.fit_transform(X_train)
Xte_char = tfidf_char.transform(X_test)

clf_char = LogisticRegression(max_iter=1000)
clf_char.fit(Xtr_char, y_train)
print("CHAR-LEVEL TF-IDF")
print(classification_report(y_test, clf_char.predict(Xte_char)))

WORD-LEVEL TF-IDF
              precision    recall  f1-score   support

           0       0.99      0.97      0.98       134
           1       0.97      0.99      0.98       134

    accuracy                           0.98       268
   macro avg       0.98      0.98      0.98       268
weighted avg       0.98      0.98      0.98       268

CHAR-LEVEL TF-IDF
              precision    recall  f1-score   support

           0       1.00      0.98      0.99       134
           1       0.98      1.00      0.99       134

    accuracy                           0.99       268
   macro avg       0.99      0.99      0.99       268
weighted avg       0.99      0.99      0.99       268



In [10]:
preds = clf_char.predict(Xte_char)
wrong = X_test[preds != y_test.values]
print(wrong.tolist())

['keep it up bro', 'mast joke maara', 'bakchodi mat kar']


In [11]:
!pip install transformers datasets -q

from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
from datasets import Dataset
import numpy as np
from sklearn.metrics import accuracy_score, f1_score

model_name = "xlm-roberta-base"
tokenizer = AutoTokenizer.from_pretrained(model_name)

train_ds = Dataset.from_dict({'text': X_train.tolist(), 'label': y_train.tolist()})
test_ds = Dataset.from_dict({'text': X_test.tolist(), 'label': y_test.tolist()})

def tokenize_fn(batch):
    return tokenizer(batch['text'], padding='max_length', truncation=True, max_length=64)

train_ds = train_ds.map(tokenize_fn, batched=True)
test_ds = test_ds.map(tokenize_fn, batched=True)

model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2)

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=1)
    return {
        'accuracy': accuracy_score(labels, preds),
        'f1': f1_score(labels, preds)
    }

args = TrainingArguments(
    output_dir="./results",
    num_train_epochs=4,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    eval_strategy="epoch",
    save_strategy="no",
    learning_rate=2e-5,
    logging_steps=10,
)

trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_ds,
    eval_dataset=test_ds,
    compute_metrics=compute_metrics,
)

trainer.train()
print(trainer.evaluate())

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:124: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

...

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] XLMRobertaForSequenceClassification LOAD REPORT from: xlm-roberta-base
Key                         | Status     | 
----------------------------+------------+-
lm_head.dense.weight        | UNEXPECTED | 
lm_head.layer_norm.weight   | UNEXPECTED | 
lm_head.dense.bias          | UNEXPECTED | 
lm_head.layer_norm.bias     | UNEXPECTED | 
lm_head.bias                | UNEXPECTED | 
roberta.pooler.dense.bias   | UNEXPECTED | 
roberta.pooler.dense.weight | UNEXPECTED | 
classifier.out_proj.bias    | MISSING    | 
classifier.dense.weight     | MISSING    | 
classifier.dense.bias       | MISSING    | 
classifier.out_proj.weight  | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Epoch,Training Loss,Validation Loss,Accuracy,F1
1,0.118877,0.135103,0.973881,0.973783
2,0.086171,0.063801,0.988806,0.988930
3,0.043574,0.081084,0.988806,0.988930
4,0.007741,0.048483,0.992537,0.992593


Training Loss,Validation Loss,Epoch,Accuracy,F1
0.007741,0.048483,4,0.992537,0.992593


{'eval_loss': 0.04848342761397362, 'eval_accuracy': 0.9925373134328358, 'eval_f1': 0.9925925925925926}


In [12]:
preds = trainer.predict(test_ds).predictions.argmax(axis=1)
wrong = np.where(preds != np.array(y_test))[0]
print(f"Number of mistakes: {len(wrong)}")
for i in wrong:
    print(X_test.iloc[i], "| true:", y_test.iloc[i], "| pred:", preds[i])


Number of mistakes: 2
mast joke maara | true: 0 | pred: 1
bakchodi mat kar | true: 0 | pred: 1


In [21]:
test_sentences = [
    "yaar tu kitna ghatiya insaan hai",
    "bhai mast match tha aaj",
    "teri shakal dekh ke hi pata chal jata hai",
    "chal hatt yaha se bsdk",
    "Chill bhai kya kar raha hai",
    "Abbe jaana chutiye",
    "bhag gandu ma chuda jaake",
    "Yo i love that pizza bhai it is so good",
    "Yeah I know usko farak nahi padta bhai",
    "Uske mama ka bh0s*a uski ma k! ch!t"
]

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)

inputs = tokenizer(test_sentences, padding=True, truncation=True, max_length=64, return_tensors='pt').to(device)
outputs = model(**inputs)
preds_real = outputs.logits.argmax(dim=1)
for s, p in zip(test_sentences, preds_real):
    print(s, "->", "offensive" if p.item()==1 else "not offensive")

yaar tu kitna ghatiya insaan hai -> not offensive
bhai mast match tha aaj -> not offensive
teri shakal dekh ke hi pata chal jata hai -> not offensive
chal hatt yaha se bsdk -> offensive
Chill bhai kya kar raha hai -> not offensive
Abbe jaana chutiye -> offensive
bhag gandu ma chuda jaake -> offensive
Yo i love that pizza bhai it is so good -> not offensive
Yeah I know usko farak nahi padta bhai -> not offensive
Uske mama ka bh0s*a uski ma k! ch!t -> offensive


In [22]:
X_custom_word = tfidf_word.transform(test_sentences)
word_preds = clf_word.predict(X_custom_word)

X_custom_char = tfidf_char.transform(test_sentences)
char_preds = clf_char.predict(X_custom_char)

print("Word TF-IDF model prediction:",word_preds)
print("Char TF-IDF model prediction:",char_preds)

Word TF-IDF model prediction: [0 0 1 1 0 1 1 0 0 1]
Char TF-IDF model prediction: [0 0 0 1 0 1 1 0 0 1]


In [29]:
from google.colab import _message

# 1. Get the current live notebook
res = _message.blocking_request('get_ipynb', request='', timeout_sec=5)
nb = res['ipynb']

# 2. Define problematic keys
problematic_keys = ['widgets', 'user_expressions', 'colab_outputs_summary']

# 3. Clean root metadata
if 'metadata' in nb:
    for key in problematic_keys:
        nb['metadata'].pop(key, None)
    # Also clean specific colab internal metadata that can cause issues
    if 'colab' in nb['metadata']:
        nb['metadata']['colab'].pop('base_uri', None)

# 4. Clean every single cell's metadata
for cell in nb.get('cells', []):
    if 'metadata' in cell:
        for key in problematic_keys:
            cell['metadata'].pop(key, None)
        if 'colab' in cell['metadata']:
            cell['metadata']['colab'].pop('base_uri', None)

# 5. Push the cleaned version back to the session
_message.blocking_request('set_ipynb', request={'ipynb': nb}, timeout_sec=5)

print("Deep clean complete. Please try 'File > Save a copy in GitHub' now.")

Deep clean complete. Please try 'File > Save a copy in GitHub' now.
